In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit import RDLogger
from rdkit.Chem import AllChem
from rdkit.Chem import rdFingerprintGenerator
import csv
import h5py
import concurrent.futures
from rdkit.Chem import Descriptors
from collections import Counter
import itertools

In [ ]:
chunk_size = 50000

tsv_path = '______________PATH_________________'    # PUT PATH HERE

cols = ['Ligand SMILES', 'Ki (nM)', 'BindingDB Target Chain Sequence 1']

In [ ]:
# Function to extract smiles and make morgan fingerprints along with extractiong protein data.

def process_molecule(row_data):
    smiles, ki, sequence = row_data
    mol = Chem.MolFromSmiles(smiles)
    if mol and ki > 0:
        # Making Morgan Fingerprints
        fp = mfpgen.GetFingerprint(mol)
        pki = -np.log10(ki * 1e-9)
        # Extracting Descriptors
        mw = Descriptors.ExactMolWt(mol)
        logp = Descriptors.MolLogP(mol)
        tpsa = Descriptors.TPSA(mol)
        hbd = Descriptors.NumHDonors(mol)
        hba = Descriptors.NumHAcceptors(mol)
        fp_enriched = np.append(fp, [mw, logp, tpsa, hbd, hba])
        # Protein Chain Extraction
        amino_acids = "ACDEFGHIKLMNPQRSTVWY"
        seq_len = len(sequence) if len(sequence) > 0 else 1
        counts = Counter(sequence)
        dipeptides = [''.join(pair) for pair in itertools.product(amino_acids, repeat=2)]
    
        seq = str(sequence).strip().upper()
        
        total_pairs = len(seq) - 1 if len(seq) > 1 else 1 
        # Finding Dipetides to figure out alpha helix and beta pleated sheets.
        counts = {dp: 0 for dp in dipeptides}
        for i in range(len(seq) - 1):
            pair = seq[i:i+2]
            if pair in counts:
                counts[pair] += 1
        return np.array(fp_enriched, dtype = np.float32), [pki],np.array([counts[dp] / total_pairs for dp in dipeptides], dtype=np.float32)
    return None

In [ ]:
with h5py.File('binding_data_cl.h5', 'w') as h5f_lig, h5py.File('Protein_data_cl.h5', 'w') as h5f_prot :
    x_dset = h5f_lig.create_dataset('features', shape = (0, 2053), maxshape = (None, 2053), dtype = 'int8', chunks = True)  # Using int8 to save space as all of the data is binary.
    y_targ = h5f_lig.create_dataset('target', shape = (0, 1), maxshape = (None, 1), dtype = 'float32', chunks = True)
    prot = h5f_prot.create_dataset('BindingDB Target Chain Sequence 1', shape = (0, 400), maxshape = (None, 400), dtype = 'float32', chunks = True)
    RDLogger.DisableLog('rdApp.*')
    mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius = 2, fpSize = 2048)        # Morgan Fingerprint Generator
    with concurrent.futures.ProcessPoolExecutor() as executor:              # For faster processing using all CPU cores
        for chunk in pd.read_csv(tsv_path, sep = '\t', usecols = cols, chunksize = chunk_size, on_bad_lines = 'skip', engine = 'python', quoting = csv.QUOTE_NONE):
            chunk = chunk.dropna(subset=['Ligand SMILES', 'BindingDB Target Chain Sequence 1', 'Ki (nM)'])
            chunk['Ki (nM)'] = chunk['Ki (nM)'].astype(str).str.replace('>', ' ').str.replace('<', ' ').astype(float)
            chunk = chunk.groupby(['Ligand SMILES', 'BindingDB Target Chain Sequence 1'], as_index=False)['Ki (nM)'].median()
            data_to_process = list(zip(chunk['Ligand SMILES'], chunk['Ki (nM)'], chunk['BindingDB Target Chain Sequence 1']))  # Zipping files to feed in function
            x_chunk = []
            y_chunk = []
            p_chunk = []
            results = list(executor.map(process_molecule, data_to_process))
            for res in results:
                if res is not None:
                    x_chunk.append(res[0])
                    y_chunk.append(res[1])
                    p_chunk.append(res[2])
            if len(x_chunk) > 0:
                x_arr = np.array(x_chunk, dtype = np.float32)
                y_arr = np.array(y_chunk, dtype = np.float32)
                p_arr = np.array(p_chunk, dtype = np.float32)
                current_size = x_dset.shape[0]
                new_size = current_size + x_arr.shape[0]        # Resizing
                x_dset.resize(new_size, axis = 0)
                y_targ.resize(new_size, axis = 0)
                prot.resize(new_size, axis = 0)

                x_dset[current_size:new_size, :] = x_arr
                y_targ[current_size:new_size, :] = y_arr
                prot[current_size:new_size, :] = p_arr
                print(f"Processed up to row {new_size}....")


Processed up to row 15099....
Processed up to row 18443....
Processed up to row 19649....
Processed up to row 37547....
Processed up to row 47177....
Processed up to row 54964....
Processed up to row 64794....
Processed up to row 72616....
Processed up to row 81621....
Processed up to row 85466....
Processed up to row 88381....
Processed up to row 94902....
Processed up to row 101649....
Processed up to row 107113....
Processed up to row 111724....
Processed up to row 116077....
Processed up to row 120009....
Processed up to row 124585....
Processed up to row 127624....
Processed up to row 134915....
Processed up to row 140489....
Processed up to row 145080....
Processed up to row 148517....
Processed up to row 151451....
Processed up to row 155224....
Processed up to row 163785....
Processed up to row 167697....
Processed up to row 170311....
Processed up to row 173434....
Processed up to row 179440....
Processed up to row 184214....
Processed up to row 190038....
Processed up to row 